# Restore MAGFiLO to the new Google Drive

Run **Runtime → Run all** for a fresh session. If the extracted dataset already exists in this Colab runtime, the notebook checks 887 images and the annotations JSON and **reuses it without downloading again**.

**After any upload interruption**, run **only the final cell** (`RESUME UPLOAD ONLY`); it checks files already on the Colab filesystem and uploads missing pieces without re-downloading. If you already prepared the manifest and four pieces before a download failure, you can run this final cell directly.

Only split parts and the checksum manifest are uploaded to Drive. No credentials are written to GitHub or Drive.



In [ ]:
%pip -q install -U kagglehub
from google.colab import auth
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
creds, _ = google.auth.default()
drive_api = build('drive','v3',credentials=creds,cache_discovery=False)
FOLDER_ID = "1aA5QZgdnJVJdiudUAJWh2yoQN65oNmka"
target = drive_api.files().get(fileId=FOLDER_ID,fields='id,name,mimeType').execute()
assert target['mimeType']=='application/vnd.google-apps.folder', 'The Drive destination is not a folder.'
print('Verified the new Drive folder:',target['name'],target['id'])


In [ ]:
# Steps 2–4 run in one cell to prevent NameError when cells are executed out of order.
if 'drive_api' not in globals() or 'FOLDER_ID' not in globals():
    raise RuntimeError('First run the authentication cell immediately above this cell (Runtime > Run all).')

import os
from pathlib import Path
from getpass import getpass
import kagglehub
from kagglehub.exceptions import UnauthenticatedError

RULES = "https://www.kaggle.com/competitions/filament-segmentation-2026/rules"
SETTINGS = "https://www.kaggle.com/settings/api"

def authenticate_kaggle(token):
    token = token.strip()
    # Accept both a bare token and the full command copied from Kaggle settings.
    if token.startswith("export KAGGLE_API_TOKEN="):
        token = token.split("=", 1)[1].strip()
    elif token.startswith("KAGGLE_API_TOKEN="):
        token = token.split("=", 1)[1].strip()
    if len(token) >= 2 and token[0] == token[-1] and token[0] in ("'", '"'):
        token = token[1:-1].strip()
    if token.startswith("{") or token.lower().endswith(".json"):
        raise ValueError(
            "A kaggle.json file or JSON was provided, not a Kaggle API token. "
            "Use 'Generate New Token' on your Kaggle API settings page, "
            "then paste ONLY its token value."
        )
    if not token:
        raise ValueError("No API token provided.")
    # Kaggle's modern API tokens normally begin with KGAT_.
    os.environ["KAGGLE_API_TOKEN"] = token
    kagglehub.config.set_kaggle_api_token(token)
    try:
        identity = kagglehub.whoami(verbose=False)
    except Exception as exc:
        raise RuntimeError(
            "Kaggle could not validate this token. Check that the value is the "
            "API token (not a username, account password, or kaggle.json file), "
            "and generate it in the Kaggle account joined to the competition: " + SETTINGS
        ) from exc
    username = identity.get("username")
    if not username:
        raise RuntimeError("Kaggle returned no authenticated username.")
    print("Kaggle identity VERIFIED:", username)
    return username

token = os.environ.get("KAGGLE_API_TOKEN", "").strip()
if not token:
    try:
        from google.colab import userdata
        token = (userdata.get("KAGGLE_API_TOKEN") or "").strip()
    except Exception:
        token = ""

print("Google Drive identity is separate from the Kaggle competition account.")
if not token:
    print("Open the API settings in the Kaggle account registered for this competition:", SETTINGS)
    token = getpass("Paste that account's NEW API token (hidden): ")
try:
    username = authenticate_kaggle(token)
except Exception as initial_auth_error:
    print("Kaggle authentication was not verified. Error:", type(initial_auth_error).__name__)
    print("If a token was cached from another account, generate a new token from the")
    print("Kaggle account that joined this competition. The token will not be printed.")
    replacement = getpass("Try a fresh competition-account token (hidden), or Enter to stop: ").strip()
    if not replacement:
        raise RuntimeError("Authentication failed. No valid Kaggle token available.") from initial_auth_error
    username = authenticate_kaggle(replacement)
finally:
    token = None

work = Path("/content/magfilo_kaggle_download")
work.mkdir(exist_ok=True)

def _complete_extracted_dataset(folder):
    """Reuse only a clearly complete MAGFiLO extraction, never a partial download."""
    import json
    if not folder.is_dir():
        return False
    candidates = [
        p for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in {".jpeg", ".jpg"}
    ]
    train = [p for p in candidates if "train_images" in p.parts]
    test = [p for p in candidates if "test_images" in p.parts]
    annotations = [
        p for p in folder.rglob("*.json")
        if p.is_file() and "annotations" in p.name.lower()
    ]
    # The competition archive previously verified: 887 images and 1 annotations JSON.
    if len(train) + len(test) != 887 or not train or not test or len(annotations) != 1:
        return False
    try:
        for p in train + test:
            if p.stat().st_size < 100:
                return False
            with p.open("rb") as stream:
                if stream.read(2) != b"\xff\xd8":
                    return False
                stream.seek(-2, 2)
                if stream.read(2) != b"\xff\xd9":
                    return False
        with annotations[0].open("r", encoding="utf-8") as stream:
            parsed = json.load(stream)
        return isinstance(parsed.get("images"), list) and isinstance(parsed.get("annotations"), list)
    except (OSError, ValueError, UnicodeError):
        return False

if _complete_extracted_dataset(work):
    downloaded = work
    print("REUSING verified existing MAGFiLO images (887 images + annotations).")
else:
    if any(work.iterdir()):
        # Never overwrite or purge the previous incomplete output.
        for attempt_number in range(1, 30):
            fresh = work.with_name(f"{work.name}_retry_{attempt_number:02d}")
            if not fresh.exists() or not any(fresh.iterdir()):
                fresh.mkdir(parents=True, exist_ok=True)
                work = fresh
                break
        else:
            raise RuntimeError("No empty download directory is available in /content.")
        print("Earlier output appears incomplete; preserving it and downloading into:", work)
    for attempt in range(2):
        try:
            downloaded = Path(kagglehub.competition_download(
                "filament-segmentation-2026", output_dir=str(work)
            ))
            break
        except Exception as exc:
            if "401" not in str(exc) and "403" not in str(exc) and not isinstance(exc, UnauthenticatedError):
                raise
            print("Kaggle returned access denied for authenticated user:", username)
            print("Verify this EXACT user joined the competition:", RULES)
            if attempt == 0:
                print("If you joined from another Kaggle account, generate that account's token.")
                replacement_token = getpass(
                    "Competition-account token to retry once (hidden), or Enter to stop: "
                ).strip()
                if replacement_token:
                    username = authenticate_kaggle(replacement_token)
                    replacement_token = None
                    continue
            raise RuntimeError(
                "401/403 from Kaggle: the authenticated user lacks competition access "
                "or the rules/verification are not complete. Open the competition rules "
                "in the Kaggle account printed above before retrying."
            ) from exc

if not downloaded.exists():
    raise FileNotFoundError(f'Kaggle download returned a nonexistent path: {downloaded}')
print('Downloaded:', downloaded)

import zipfile, shutil
def valid_zip(p):
    try:
        with zipfile.ZipFile(p) as z:
            names=[x.lower().replace('\\','/') for x in z.namelist()]
        return (any('train_images/' in n and n.endswith(('.jpg','.jpeg')) for n in names)
                and any('test_images/' in n and n.endswith(('.jpg','.jpeg')) for n in names)
                and any('annotations' in n and n.endswith('.json') for n in names))
    except (OSError,zipfile.BadZipFile):
        return False

candidates=[downloaded] if downloaded.is_file() else list(downloaded.rglob('*'))
source_zip=next((p for p in candidates if p.is_file() and p.suffix.lower()=='.zip' and valid_zip(p)),None)
archive=Path('/content/filament-segmentation-2026.zip')
if source_zip:
    if source_zip.resolve()!=archive.resolve(): shutil.copyfile(source_zip,archive)
else:
    files=[p for p in candidates if p.is_file() and p.suffix.lower() in ('.jpg','.jpeg','.json')]
    if not files: raise RuntimeError('No images or JSON were downloaded.')
    base=downloaded if downloaded.is_dir() else downloaded.parent
    with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_STORED,allowZip64=True) as z:
        for p in sorted(files): z.write(p,p.relative_to(base).as_posix())
if not valid_zip(archive): raise RuntimeError('Train/test images or annotations missing; upload aborted.')
with zipfile.ZipFile(archive) as z:
    print('Verified',sum(n.endswith(('.jpg','.jpeg')) for n in z.namelist()),'images and',sum(n.endswith('.json') for n in z.namelist()),'JSON files.')

import hashlib,math,json
from googleapiclient.http import MediaFileUpload
PART_SIZE=180*1024*1024
BUFFER=8*1024*1024
def sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as src:
        for data in iter(lambda:src.read(BUFFER),b''): h.update(data)
    return h.hexdigest()
files_to_upload=[] # Keep only accessible <256-MiB parts in Drive; avoid duplicated archive
parts=[]
with archive.open('rb') as source:
    for idx in range(1,math.ceil(archive.stat().st_size/PART_SIZE)+1):
        path=Path('/content') / (archive.name+f'.part{idx:02d}')
        remaining=min(PART_SIZE,archive.stat().st_size-(idx-1)*PART_SIZE)
        with path.open('wb') as out:
            while remaining:
                chunk=source.read(min(BUFFER,remaining))
                if not chunk: raise IOError('Unexpected end of ZIP')
                out.write(chunk);remaining-=len(chunk)
        parts.append({'name':path.name,'bytes':path.stat().st_size,'sha256':sha256(path)})
        files_to_upload.append(path)
manifest={'archive':archive.name,'bytes':archive.stat().st_size,'sha256':sha256(archive),'parts':parts}
manifest_file=Path('/content/filament-segmentation-2026.parts.json')
manifest_file.write_text(json.dumps(manifest,indent=2),encoding='utf-8')
files_to_upload.append(manifest_file)


In [ ]:
# RESUME UPLOAD ONLY: this cell does NOT download from Kaggle again.
# It can be run separately after the preparation cell, including after an interrupted upload.
from pathlib import Path
import json
import hashlib
from googleapiclient.http import MediaFileUpload

if "drive_api" not in globals() or "FOLDER_ID" not in globals():
    raise RuntimeError("Run the Google Drive authorization cell first.")

manifest_path = Path("/content/filament-segmentation-2026.parts.json")
if not manifest_path.is_file():
    raise FileNotFoundError(
        "No prepared manifest in /content. Run the data download/preparation cell first."
    )

data = json.loads(manifest_path.read_text(encoding="utf-8"))
upload_files = []
for entry in data["parts"]:
    part = manifest_path.parent / entry["name"]
    if not part.is_file() or part.stat().st_size != entry["bytes"]:
        raise RuntimeError(f"Missing or incomplete dataset part: {part}")
    upload_files.append(part)
upload_files.append(manifest_path)

def _md5(path):
    h = hashlib.md5()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

for index, path in enumerate(upload_files, 1):
    # Drive uses pageSize (camel-case), not page_size.
    escaped = path.name.replace("\\", "\\\\").replace("'", "\\'")
    query = f"'{FOLDER_ID}' in parents and name = '{escaped}' and trashed = false"
    response = drive_api.files().list(
        q=query, fields="nextPageToken,files(id,name,size,md5Checksum)", pageSize=100
    ).execute()
    existing = response.get("files", [])
    if response.get("nextPageToken"):
        raise RuntimeError(f"More than 100 matching Drive files for {path.name}; refusing to guess.")
    expected_md5 = _md5(path)
    if existing:
        if len(existing) == 1 and existing[0].get("md5Checksum") == expected_md5:
            print(f"[{index}/{len(upload_files)}] Already uploaded & checksum verified: {path.name}")
            continue
        raise FileExistsError(f"Conflicting file already present in Drive: {path.name}")

    media = MediaFileUpload(
        str(path), mimetype="application/octet-stream" if path != manifest_path else "application/json",
        chunksize=8 * 1024 * 1024, resumable=True
    )
    request = drive_api.files().create(
        body={"name": path.name, "parents": [FOLDER_ID]},
        media_body=media, fields="id,name,size,md5Checksum"
    )
    result = None
    while result is None:
        status, result = request.next_chunk()
    if result.get("md5Checksum") != expected_md5:
        raise RuntimeError(f"Drive checksum mismatch after upload: {path.name}")
    print(f"[{index}/{len(upload_files)}] Saved and verified: {result['name']}")

print("FINISHED: parts and manifest verified in the NEW Google Drive.")
